# 🔍 Legal Chunker — Visual Debug Pipeline

**Mục tiêu**: Chạy `ClauseChunker` trên một file `.md` cụ thể và hiển thị
từng bước để dễ dàng kiểm tra chất lượng output.

**Luồng**:
```
.md file → State Machine Parser → ParentChunk (Điều) + ChildChunk (Khoản) → JSONL
```

**Hướng dẫn**: Chỉ cần thay `TARGET_FILE` ở Cell 1 rồi chạy `Run All`.

In [6]:
import os, sys, json, re, shutil, tempfile
from pathlib import Path
from IPython.display import display, HTML, Markdown

# ── CẤU HÌNH — chỉnh tại đây ─────────────────────────────────────
PROJECT_ROOT = Path('d:/GenAI/AIAgent/Agentic-RAG-Traffic-Law').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# File muốn debug (đặt trong data/processed_example/)
TARGET_FILE  = PROJECT_ROOT / 'data' / 'processed_example' / '172475.md'
TITLE_JSON   = PROJECT_ROOT / 'outputs' / 'traffic_law_filtered_by_title.json'

print(f'✅ PROJECT_ROOT : {PROJECT_ROOT}')
print(f'✅ TARGET_FILE  : {TARGET_FILE} (exists={TARGET_FILE.exists()})')
print(f'✅ TITLE_JSON   : {TITLE_JSON} (exists={TITLE_JSON.exists()})')


✅ PROJECT_ROOT : D:\GenAI\AIAgent\Agentic-RAG-Traffic-Law
✅ TARGET_FILE  : D:\GenAI\AIAgent\Agentic-RAG-Traffic-Law\data\processed_example\172475.md (exists=True)
✅ TITLE_JSON   : D:\GenAI\AIAgent\Agentic-RAG-Traffic-Law\outputs\traffic_law_filtered_by_title.json (exists=True)


## 📄 BƯỚC 1 — Xem nội dung file gốc

In [7]:
content = TARGET_FILE.read_text(encoding='utf-8')
lines   = content.split('\n')

print(f'📄 File: {TARGET_FILE.name}')
print(f'📏 Tổng số dòng : {len(lines)}')
print(f'📦 Kích thước   : {len(content):,} bytes')
print('\n' + '─'*70)
print('👀 100 dòng đầu tiên:')
print('─'*70)
for i, ln in enumerate(lines[:100]):
    print(f'{i+1:4d} │ {ln}')


📄 File: 172475.md
📏 Tổng số dòng : 1853
📦 Kích thước   : 148,089 bytes

──────────────────────────────────────────────────────────────────────
👀 100 dòng đầu tiên:
──────────────────────────────────────────────────────────────────────
   1 │ QUỐC HỘI _______ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc _______________________ Luật số: 35/2024/QH15
   2 │ 
   3 │ LUẬT
   4 │ 
   5 │ Đường bộ
   6 │ 
   7 │ Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;
   8 │ 
   9 │ Quốc hội ban hành Luật Đường bộ.
  10 │ 
  11 │ Chương I
  12 │ 
  13 │ NHỮNG QUY ĐỊNH CHUNG
  14 │ 
  15 │ Điều 1. Phạm vi điều chỉnh
  16 │ 
  17 │ Luật này quy định về hoạt động đường bộ và quản lý nhà nước về hoạt động đường bộ.
  18 │ 
  19 │ Điều 2. Giải thích từ ngữ
  20 │ 
  21 │ Trong Luật này, các từ ngữ dưới đây được hiểu như sau:
  22 │ 
  23 │ 1. Hoạt động đường bộ bao gồm: hoạt động về quy hoạch, đầu tư, xây dựng, quản lý, sử dụng, vận hành, khai thác, bảo trì, bảo vệ kết cấu hạ tầ

## 🔎 BƯỚC 2 — Kiểm tra Regex patterns khớp với file

In [8]:
RE_CHAPTER = re.compile(r'^CHƯƠNG\s+([IXVLCDM]+)[\s.]*(.*)$', re.IGNORECASE)
RE_SECTION = re.compile(r'^Mục\s+(\d+)[\s.]*(.*)$', re.IGNORECASE)
RE_ARTICLE = re.compile(r'^Điều\s+(\d+[a-zđA-ZĐ]?)[\.:] *(.*)$', re.IGNORECASE)
RE_CLAUSE  = re.compile(r'^(\d+)\.\s+(.+)$')

hits = {'CHƯƠNG': [], 'Mục': [], 'Điều': [], 'Khoản': []}

for i, ln in enumerate(lines):
    s = ln.strip()
    if RE_CHAPTER.match(s): hits['CHƯƠNG'].append((i+1, s))
    elif RE_SECTION.match(s): hits['Mục'].append((i+1, s))
    elif RE_ARTICLE.match(s): hits['Điều'].append((i+1, s))
    elif RE_CLAUSE.match(s):  hits['Khoản'].append((i+1, s))

for key, lst in hits.items():
    print(f'\n📌 {key} — {len(lst)} dòng khớp:')
    for lineno, text in lst[:8]:
        print(f'   dòng {lineno:4d}: {text[:90]}')
    if len(lst) > 8:
        print(f'   ... và {len(lst)-8} dòng khác')



📌 CHƯƠNG — 6 dòng khớp:
   dòng   11: Chương I
   dòng  127: Chương II
   dòng 1097: Chương III
   dòng 1315: Chương IV
   dòng 1751: Chương V
   dòng 1793: Chương VI

📌 Mục — 4 dòng khớp:
   dòng  131: Mục 1
   dòng  221: Mục 2
   dòng  523: Mục 3
   dòng  729: Mục 4

📌 Điều — 86 dòng khớp:
   dòng   15: Điều 1. Phạm vi điều chỉnh
   dòng   19: Điều 2. Giải thích từ ngữ
   dòng   37: Điều 3. Nguyên tắc hoạt động đường bộ
   dòng   47: Điều 4. Chính sách phát triển đối với hoạt động đường bộ
   dòng   61: Điều 5. Quy hoạch mạng lưới đường bộ, quy hoạch kết cấu hạ tầng đường bộ
   dòng   95: Điều 6. Cơ sở dữ liệu đường bộ
   dòng  113: Điều 7. Các hành vi bị nghiêm cấm
   dòng  135: Điều 8. Phân loại đường bộ theo cấp quản lý
   ... và 78 dòng khác

📌 Khoản — 388 dòng khớp:
   dòng   23: 1. Hoạt động đường bộ bao gồm: hoạt động về quy hoạch, đầu tư, xây dựng, quản lý, sử dụng,
   dòng   25: 2. Đường bộ bao gồm: đường, cầu đường bộ, cống đường bộ, hầm đường bộ, bến phà đường bộ, c
   dò

## ⚙️ BƯỚC 3 — Chạy ClauseChunker

In [11]:
from chunking.legal_chunker import ClauseChunker

with tempfile.TemporaryDirectory() as td:
    tp  = Path(td)
    shutil.copy(TARGET_FILE, tp / TARGET_FILE.name)
    out = tp / 'chunks.jsonl'

    chunker = ClauseChunker(
        input_dir=str(tp),
        output_file=str(out),
        title_json_path=str(TITLE_JSON) if TITLE_JSON.exists() else None
    )
    chunker.run()

    ALL_CHUNKS = []
    with open(out, encoding='utf-8') as f:
        for ln in f:
            if ln.strip(): ALL_CHUNKS.append(json.loads(ln))

PARENTS  = [c for c in ALL_CHUNKS if c['type'] == 'parent']
CHILDREN = [c for c in ALL_CHUNKS if c['type'] == 'child']
UNSTRUCT = [c for c in ALL_CHUNKS if c['type'] == 'unstructured']

print("=============================================")
print(f' Tổng chunks    : {len(ALL_CHUNKS)}')
print(f' Parent (Điều)  : {len(PARENTS)}')
print(f' Child  (Khoản) : {len(CHILDREN)}')
print(f' Unstructured   : {len(UNSTRUCT)}')
print("=============================================")


🚀 ClauseChunker — Hierarchical State Machine (Điều/Khoản)
📂 Nguồn: C:\Users\phatt\AppData\Local\Temp\tmp1csjh5s5 | Đích: C:\Users\phatt\AppData\Local\Temp\tmp1csjh5s5\chunks.jsonl
------------------------------------------------------------

🎉 HOÀN TẤT! Files: 1 | Chunks: 474 | Lỗi: 0

📊 BÁO CÁO CHẤT LƯỢNG:
{
  "total_chunks": 474,
  "article_level_UNKNOWN_ratio": "0.00%",
  "clause_level_UNKNOWN_ratio": "0.00%",
  "doc_types": {
    "parent": 86,
    "child": 388
  }
}
 Tổng chunks    : 474
 Parent (Điều)  : 86
 Child  (Khoản) : 388
 Unstructured   : 0


## 📊 BƯỚC 4 — Tổng quan phân phối

In [12]:
from collections import Counter

# Chapter distribution
chap_dist = Counter(c['metadata'].get('chapter','?') for c in PARENTS)
print('📌 Số Điều theo Chương:')
for ch, cnt in sorted(chap_dist.items(), key=lambda x: -x[1]):
    bar = '█' * min(cnt, 40)
    print(f'  {ch[:55]:<55} {bar} {cnt}')

# Children per parent
parent_id_map = {p['chunk_id']: p for p in PARENTS}
child_counts  = Counter(c['parent_id'] for c in CHILDREN)
print(f'\n📌 Số Khoản / Điều:')
print(f'  Max : {max(child_counts.values(), default=0)}')
print(f'  Min : {min(child_counts.values(), default=0)}')
avg = sum(child_counts.values())/len(child_counts) if child_counts else 0
print(f'  Avg : {avg:.1f}')
print(f'  Điều không có Khoản: {len(PARENTS)-len(child_counts)}')


📌 Số Điều theo Chương:
  Chương II: KẾT CẤU HẠ TẦNG ĐƯỜNG BỘ                     ████████████████████████████████████ 36
  Chương IV: VẬN TẢI ĐƯỜNG BỘ                             █████████████████████████ 25
  Chương III: ĐƯỜNG BỘ CAO TỐC                            ████████████ 12
  Chương I: NHỮNG QUY ĐỊNH CHUNG                          ███████ 7
  Chương V: QUẢN LÝ NHÀ NƯỚC VỀ HOẠT ĐỘNG ĐƯỜNG BỘ        ███ 3
  Chương VI: ĐIỀU KHOẢN THI HÀNH                          ███ 3

📌 Số Khoản / Điều:
  Max : 14
  Min : 2
  Avg : 4.6
  Điều không có Khoản: 2


## 🌳 BƯỚC 5 — Xem cây Parent → Child (toàn bộ Điều)

In [13]:
# Thay đổi số Điều muốn xem (1-based index trong danh sách PARENTS)
VIEW_PARENT_INDICES = list(range(min(5, len(PARENTS))))  # xem 5 Điều đầu

def show_article_tree(parent_idx: int):
    if parent_idx >= len(PARENTS):
        print(f'Không có Điều thứ {parent_idx+1}'); return
    p   = PARENTS[parent_idx]
    m   = p['metadata']
    kids = [c for c in CHILDREN if c['parent_id'] == p['chunk_id']]

    print("=============================================")
    print(f'📘 PARENT #{parent_idx+1}  chunk_id={p["chunk_id"]}')
    print(f'   Chương : {m["chapter"]}')
    print(f'   Mục    : {m["section"]}')
    print(f'   Điều   : {m["article"]}')
    print(f'   Scope  : {m["doc_scope"]}')
    print(f'   full_text ({len(p["full_text"])} chars):')
    for ln in p['full_text'].split('\n')[:6]:
        print(f'      │ {ln}')
    if len(p['full_text'].split('\n')) > 6: print('      │ ...')

    print(f'\n   └─ {len(kids)} Khoản:')
    for i, k in enumerate(kids):
        preview = k['text'].split('\n')[0][:100]
        print(f'      [{i+1}] chunk_id={k["chunk_id"]}')
        print(f'           text: {preview}')
        if len(k['text']) > 100:
            print(f'           ... ({len(k["text"])} chars tổng cộng)')

for idx in VIEW_PARENT_INDICES:
    show_article_tree(idx)


📘 PARENT #1  chunk_id=2662ea63-dfda-4918-99fb-9312ce967933
   Chương : Chương I: NHỮNG QUY ĐỊNH CHUNG
   Mục    : UNKNOWN
   Điều   : Điều 1 - Phạm vi điều chỉnh
   Scope  : main
   full_text (110 chars):
      │ Điều 1. Phạm vi điều chỉnh
      │ 
      │ Luật này quy định về hoạt động đường bộ và quản lý nhà nước về hoạt động đường bộ.

   └─ 0 Khoản:
📘 PARENT #2  chunk_id=81fb1c8b-0b43-4994-b37d-3dca952e1b0a
   Chương : Chương I: NHỮNG QUY ĐỊNH CHUNG
   Mục    : UNKNOWN
   Điều   : Điều 2 - Giải thích từ ngữ
   Scope  : main
   full_text (1740 chars):
      │ Điều 2. Giải thích từ ngữ
      │ 
      │ Trong Luật này, các từ ngữ dưới đây được hiểu như sau:
      │ 
      │ 1. Hoạt động đường bộ bao gồm: hoạt động về quy hoạch, đầu tư, xây dựng, quản lý, sử dụng, vận hành, khai thác, bảo trì, bảo vệ kết cấu hạ tầng đường bộ; vận tải đường bộ.
      │ 
      │ ...

   └─ 7 Khoản:
      [1] chunk_id=54374ca8-a477-4368-892f-eb5e873969c2
           text: 1. Hoạt động đường bộ bao gồm: hoạ

## 🔬 BƯỚC 6 — Tra cứu theo số Điều cụ thể

In [15]:
# Tra cứu một Điều cụ thể theo số Điều
DIEU_SO = 4   # ← thay số Điều muốn xem

def lookup_dieu(so: int):
    target = None
    for p in PARENTS:
        art = p['metadata'].get('article', '')
        if re.search(rf'Điều\s+{so}\b', art, re.IGNORECASE):
            target = p; break
    if not target:
        print(f'❌ Không tìm thấy Điều {so}')
        return

    kids = [c for c in CHILDREN if c['parent_id'] == target['chunk_id']]
    m    = target['metadata']

    print("=============================================")
    print(f'🔍 ĐIỀU {so}  (chunk_id: {target["chunk_id"]})')
    print(f'   Chương : {m["chapter"]}')
    print(f'   Mục    : {m["section"]}')
    print(f'   Tiêu đề: {m["article"]}')
    print(f'   Scope  : {m["doc_scope"]}')
    print(f'   full_text:')
    print(target['full_text'])

    print(f'\n--- {len(kids)} KHOẢN ---')
    for i, k in enumerate(kids):
        print(f'\nKhoản #{i+1}  [parent_id={k["parent_id"]}]')
        print(k['text'])
    print('='*70)

lookup_dieu(DIEU_SO)


🔍 ĐIỀU 4  (chunk_id: ecc04022-44f1-49ef-bd26-3027a74fa900)
   Chương : Chương I: NHỮNG QUY ĐỊNH CHUNG
   Mục    : UNKNOWN
   Tiêu đề: Điều 4 - Chính sách phát triển đối với hoạt động đường bộ
   Scope  : main
   full_text:
Điều 4. Chính sách phát triển đối với hoạt động đường bộ

1. Chính sách phát triển kết cấu hạ tầng đường bộ bao gồm:

a) Tập trung các nguồn lực phát triển kết cấu hạ tầng đường bộ hiện đại, thích ứng với biến đổi khí hậu, thân thiện với môi trường; kết nối đồng bộ các tuyến đường bộ, các phương thức vận tải khác với vận tải đường bộ;

b) Xây dựng cơ chế đẩy mạnh huy động các nguồn lực xã hội, đa dạng các hình thức, phương thức đầu tư, xây dựng, quản lý, vận hành, khai thác, bảo trì kết cấu hạ tầng đường bộ;

c) Ưu tiên phát triển các tuyến đường cao tốc, các công trình, dự án đường bộ trọng điểm kết nối vùng, khu vực, đô thị lớn, trung tâm trong nước và quốc tế; kết cấu hạ tầng đường bộ vùng đồng bào dân tộc thiểu số, miền núi, hải đảo, biên giới; kết cấu hạ tầng đư

## 🧪 BƯỚC 7 — Kiểm tra liên kết Parent → Child

In [16]:
parent_ids = {p['chunk_id'] for p in PARENTS}

broken     = [c for c in CHILDREN if c['parent_id'] not in parent_ids]
orphan_p   = [p for p in PARENTS
              if not any(c['parent_id'] == p['chunk_id'] for c in CHILDREN)]

print(f'✅ Tổng Parent chunks   : {len(PARENTS)}')
print(f'✅ Tổng Child  chunks   : {len(CHILDREN)}')
print(f'🔗 Child có parent hợp lệ : {len(CHILDREN)-len(broken)}/{len(CHILDREN)}')
print(f'⚠️  Child bị broken link   : {len(broken)}')
print(f'📭 Parent không có Khoản  : {len(orphan_p)}')

if broken:
    print('\n--- Broken children ---')
    for b in broken[:5]:
        print(f'  chunk_id={b["chunk_id"]} parent_id={b["parent_id"]}')

if orphan_p:
    print('\n--- Orphan parents (Điều chỉ có text, không có Khoản) ---')
    for op in orphan_p[:8]:
        print(f'  {op["metadata"]["article"]}')
        print(f'  full_text: {op["full_text"][:150]}')
        print()


✅ Tổng Parent chunks   : 86
✅ Tổng Child  chunks   : 388
🔗 Child có parent hợp lệ : 388/388
⚠️  Child bị broken link   : 0
📭 Parent không có Khoản  : 2

--- Orphan parents (Điều chỉ có text, không có Khoản) ---
  Điều 1 - Phạm vi điều chỉnh
  full_text: Điều 1. Phạm vi điều chỉnh

Luật này quy định về hoạt động đường bộ và quản lý nhà nước về hoạt động đường bộ.

  Điều 71 - Dịch vụ hỗ trợ vận tải đường bộ
  full_text: Điều 71. Dịch vụ hỗ trợ vận tải đường bộ

Dịch vụ hỗ trợ vận tải đường bộ bao gồm: kinh doanh dịch vụ bến xe, trạm dừng nghỉ, bãi đỗ xe, đại lý vận tả



## 💾 BƯỚC 8 — Xuất JSONL ra file thật (optional)

In [17]:
# Uncomment để xuất ra file thật
OUTPUT_DIR  = PROJECT_ROOT / 'data-ingestion' / 'chunks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSONL = OUTPUT_DIR / 'hierarchical_chunks.jsonl'

real_chunker = ClauseChunker(
    input_dir=str(TARGET_FILE.parent),
    output_file=str(OUTPUT_JSONL),
    title_json_path=str(TITLE_JSON) if TITLE_JSON.exists() else None
)
real_chunker.run()
print(f'✅ Đã xuất: {OUTPUT_JSONL}')
print('⚡ Bỏ comment đoạn trên để xuất JSONL thật.')


🚀 ClauseChunker — Hierarchical State Machine (Điều/Khoản)
📂 Nguồn: D:\GenAI\AIAgent\Agentic-RAG-Traffic-Law\data\processed_example | Đích: D:\GenAI\AIAgent\Agentic-RAG-Traffic-Law\data-ingestion\chunks\hierarchical_chunks.jsonl
------------------------------------------------------------

🎉 HOÀN TẤT! Files: 4 | Chunks: 535 | Lỗi: 0

📊 BÁO CÁO CHẤT LƯỢNG:
{
  "total_chunks": 535,
  "article_level_UNKNOWN_ratio": "1.12%",
  "clause_level_UNKNOWN_ratio": "1.12%",
  "doc_types": {
    "parent": 104,
    "child": 425,
    "unstructured": 6
  }
}
✅ Đã xuất: D:\GenAI\AIAgent\Agentic-RAG-Traffic-Law\data-ingestion\chunks\hierarchical_chunks.jsonl
⚡ Bỏ comment đoạn trên để xuất JSONL thật.


## 🗂️ BƯỚC 9 — Chạy toàn bộ thư mục processed

In [ ]:
# Chạy trên toàn bộ data/processed → data-ingestion/chunks/
# Uncomment để chạy thật

# INPUT_FOLDER = PROJECT_ROOT / 'data' / 'processed'
# OUTPUT_JSONL = PROJECT_ROOT / 'data-ingestion' / 'chunks' / 'hierarchical_chunks.jsonl'
#
# full_chunker = ClauseChunker(
#     input_dir=str(INPUT_FOLDER),
#     output_file=str(OUTPUT_JSONL),
#     title_json_path=str(TITLE_JSON)
# )
# full_chunker.run()
print('⚡ Bỏ comment đoạn trên để chạy toàn bộ pipeline.')
